# SensiFake Qwen3-VL sharded throughput benchmark

Performance-only comparison of batch size 1 and 2 using one unquantized FP16 Qwen3-VL model sharded across Kaggle Tesla T4 x2. The frozen rubric, prompt, model revision, JSON schema, and zero-shot behavior remain unchanged.

## 1. Imports

Pin the Transformers and Hub versions used by the completed pilot, then import the PyTorch inference stack.

In [ ]:
%pip install -q "transformers==5.16.1" "huggingface_hub==1.29.0" accelerate

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import random
import time
from dataclasses import dataclass
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

import huggingface_hub
import pandas as pd
import torch
import transformers
from IPython.display import display
from PIL import Image, ImageOps
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration, set_seed

## 2. Globals

The model revision is pinned and never resolved from main. Paths target only the canonical OpenFake pilot in Kaggle's current mounted-dataset layout.

In [ ]:
SEED = 42
MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
MODEL_REVISION = "ebb281ec70b05090aa6165b016eac8ec08e71b17"
MODEL_DTYPE = torch.float16
EXPECTED_GOLD_SHA256 = "c783a28509ca200d37d7b9e0f79a4bdd359c3f59f16acf2a0a80759875eb573f"
EXPECTED_GOLD_ROWS = 101
EXPECTED_MANIFEST_ROWS = 600
BENCHMARK_IMAGES_PER_LEVEL = 9
EXPECTED_BENCHMARK_IMAGES = 27
EXPECTED_GPU_COUNT = 2
MAX_IMAGE_SIDE = 896
MAX_MEMORY_PER_GPU = "14GiB"
BASELINE_TOTAL_SECONDS = 3353.603
BASELINE_IMAGES = 101
BASELINE_SECONDS_PER_IMAGE = BASELINE_TOTAL_SECONDS / BASELINE_IMAGES

KAGGLE_OWNER_ROOT = Path("/kaggle/input/datasets/saracristinabasco")
SOURCE_DATASET_ROOT = KAGGLE_OWNER_ROOT / "sensifake600"
GOLD_DATASET_ROOT = KAGGLE_OWNER_ROOT / "sensifake-gold-development"
GOLD_CSV_PATH = GOLD_DATASET_ROOT / "sensitivity_annotations.csv"
PILOT_ROOT = SOURCE_DATASET_ROOT / "datasets/datasets/openfake/pilot-600"
MANIFEST_PATH = PILOT_ROOT / "manifest.jsonl"
LEGACY_ROOT = SOURCE_DATASET_ROOT / "sensifake-600"
OUTPUT_DIR = Path("/kaggle/working/qwen3vl_sharded_throughput_benchmark")
BENCHMARK_RESULTS_PATH = OUTPUT_DIR / "benchmark_results.csv"
BENCHMARK_SUMMARY_PATH = OUTPUT_DIR / "benchmark_summary.json"
CONFIG_A_PATH = OUTPUT_DIR / "config_a_predictions.csv"
CONFIG_B_PATH = OUTPUT_DIR / "config_b_predictions.csv"

GENERATION_PARAMETERS = {
    "max_new_tokens": 220,
    "do_sample": False,
    "use_cache": True,
}
BENCHMARK_CONFIGS = (
    {"config_id": "A", "batch_size": 1, "predictions_path": CONFIG_A_PATH},
    {"config_id": "B", "batch_size": 2, "predictions_path": CONFIG_B_PATH},
)

GOLD_REQUIRED_COLUMNS = (
    "content_hash", "blind_id", "public_relevance", "harm_urgency",
    "vulnerability", "sensitivity_score", "sensitivity_level",
    "sensitivity_rationale", "annotation_confidence", "needs_review",
    "annotation_round", "annotated_at",
)
PREDICTION_COLUMNS = [
    "content_hash", "sample_position", "config_id", "batch_size",
    "batch_index", "position_in_batch", "actual_batch_size",
    "pred_public_relevance", "pred_harm_urgency", "pred_vulnerability",
    "pred_sensitivity_score", "pred_sensitivity_level",
    "pred_sensitivity_rationale", "pred_annotation_confidence",
    "parse_ok", "parse_error", "raw_model_output",
    "batch_inference_seconds", "batch_seconds_per_image",
]

ANNOTATION_PROMPT = """You are annotating the semantic sensitivity of visible image content.
Judge only what is visibly supported by the image. Do not determine whether the image is real or synthetic. Do not infer the dataset source, provenance, generator, model, prompt, caption, or any unseen context. Do not infer political, medical, conflict, or emergency context unless it is visually supported. Assess the potential sensitivity if imagery with the visible content were deceptive.

Use this frozen rubric:
public_relevance: 0 = private/everyday content; 1 = public event, profession, institution, or recognizable public context; 2 = politics, elections, armed conflict, disaster response, or public health.
harm_urgency: 0 = no plausible immediate public harm; 1 = reputational harm or general public confusion; 2 = physical safety, democratic process, emergency response, or health behavior could be affected.
vulnerability: 0 = no visibly vulnerable/distressed subjects; 1 = patients, displaced people, injured persons, children in crisis, or populations under threat.

Predict only the three component values. Do not calculate or output a total score or sensitivity level. Return exactly one compact JSON object, with no Markdown and no extra keys:
{"public_relevance": 0, "harm_urgency": 0, "vulnerability": 0, "sensitivity_rationale": "short rationale using only visible evidence", "annotation_confidence": "high"}
The component values must be integers in their stated ranges. The rationale must be concise (at most 280 characters). annotation_confidence must be exactly low, medium, or high."""

random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)
NOTEBOOK_STARTED_AT = time.perf_counter()

## 3. Utils

Utilities preserve the pilot's strict parser, create true multimodal batches, verify CUDA-only sharding, and measure synchronized wall time and peak VRAM.

In [ ]:
@dataclass(frozen=True)
class BenchmarkItem:
    sample_position: int
    content_hash: str
    image: Image.Image


def utc_now() -> str:
    return datetime.now(UTC).isoformat(timespec="seconds")


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def seeded_order_key(content_hash: str, purpose: str) -> str:
    return hashlib.sha256(f"{SEED}:{purpose}:{content_hash}".encode()).hexdigest()


def derived_level(score: int) -> str:
    if score not in range(6):
        raise ValueError(f"sensitivity score outside 0..5: {score}")
    if score <= 1:
        return "low"
    if score <= 3:
        return "medium"
    return "high"


def reject_duplicate_json_keys(pairs: list[tuple[str, Any]]) -> dict[str, Any]:
    result: dict[str, Any] = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"duplicate JSON key: {key}")
        result[key] = value
    return result


def parse_model_json(raw_output: str) -> dict[str, Any]:
    expected = {
        "public_relevance", "harm_urgency", "vulnerability",
        "sensitivity_rationale", "annotation_confidence",
    }
    payload = json.loads(raw_output.strip(), object_pairs_hook=reject_duplicate_json_keys)
    if not isinstance(payload, dict):
        raise TypeError("response must be one JSON object")
    if set(payload) != expected:
        missing = sorted(expected - set(payload))
        extra = sorted(set(payload) - expected)
        raise ValueError(f"JSON keys do not match schema; missing={missing}, extra={extra}")
    ranges = {"public_relevance": range(3), "harm_urgency": range(3), "vulnerability": range(2)}
    for field, allowed in ranges.items():
        value = payload[field]
        if type(value) is not int or value not in allowed:
            raise ValueError(f"{field} must be an integer in {list(allowed)}")
    rationale = payload["sensitivity_rationale"]
    if not isinstance(rationale, str) or not rationale.strip():
        raise ValueError("sensitivity_rationale must be a non-empty string")
    if len(rationale.strip()) > 280:
        raise ValueError("sensitivity_rationale exceeds 280 characters")
    confidence = payload["annotation_confidence"]
    if confidence not in {"low", "medium", "high"}:
        raise ValueError("annotation_confidence must be low, medium, or high")
    payload["sensitivity_rationale"] = rationale.strip()
    score = payload["public_relevance"] + payload["harm_urgency"] + payload["vulnerability"]
    payload["sensitivity_score"] = score
    payload["sensitivity_level"] = derived_level(score)
    return payload


def resolve_manifest_path(record: dict[str, Any]) -> Path:
    candidates = [record.get(key) for key in ("relative_image_path", "path", "image_path", "file_path", "relative_path", "filename")]
    raw_path = next((value for value in candidates if isinstance(value, str) and value.strip()), None)
    if raw_path is None:
        raise KeyError("manifest record has no supported image-path field")
    supplied = Path(raw_path)
    if supplied.is_absolute():
        candidate = supplied
    elif supplied.parts[:1] == ("pilot-600",):
        candidate = PILOT_ROOT.parent / supplied
    else:
        candidate = PILOT_ROOT / supplied
    resolved = candidate.resolve()
    pilot_resolved = PILOT_ROOT.resolve()
    legacy_resolved = LEGACY_ROOT.resolve()
    if not resolved.is_relative_to(pilot_resolved):
        raise RuntimeError(f"path escaped canonical pilot root: {resolved}")
    if resolved.is_relative_to(legacy_resolved):
        raise RuntimeError(f"legacy path is forbidden: {resolved}")
    return resolved


def load_image_in_memory(path: Path) -> Image.Image:
    with Image.open(path) as source:
        image = ImageOps.exif_transpose(source).convert("RGB")
    if max(image.size) > MAX_IMAGE_SIDE:
        image.thumbnail((MAX_IMAGE_SIDE, MAX_IMAGE_SIDE), Image.Resampling.LANCZOS)
    return image


def normalize_device(value: Any) -> str:
    if isinstance(value, int):
        return f"cuda:{value}"
    text = str(value)
    if text.isdigit():
        return f"cuda:{text}"
    if text == "cuda":
        return "cuda:0"
    return text


def verify_sharded_model(model: Any) -> dict[str, str]:
    device_map = getattr(model, "hf_device_map", None)
    if not isinstance(device_map, dict) or not device_map:
        raise RuntimeError("STOP: model has no effective hf_device_map")
    effective_map = {str(name): normalize_device(device) for name, device in device_map.items()}
    mapped_devices = set(effective_map.values())
    forbidden = {device for device in mapped_devices if device in {"cpu", "disk", "meta"} or device.startswith("offload")}
    if forbidden:
        raise RuntimeError(f"STOP: CPU/disk/meta offload is forbidden: {sorted(forbidden)}")
    if mapped_devices != {"cuda:0", "cuda:1"}:
        raise RuntimeError(f"STOP: model must use both and only cuda:0/cuda:1, got {sorted(mapped_devices)}")
    parameter_devices = {str(parameter.device) for parameter in model.parameters()}
    if parameter_devices != {"cuda:0", "cuda:1"}:
        raise RuntimeError(f"STOP: parameters are not wholly sharded across both GPUs: {sorted(parameter_devices)}")
    floating_dtypes = {parameter.dtype for parameter in model.parameters() if parameter.is_floating_point()}
    if floating_dtypes != {torch.float16}:
        raise RuntimeError(f"STOP: all floating parameters must be FP16, got {floating_dtypes}")
    if getattr(model, "is_quantized", False):
        raise RuntimeError("STOP: quantized model loading is forbidden")
    if getattr(model, "quantization_method", None) is not None:
        raise RuntimeError("STOP: a quantization method was detected")
    if getattr(model, "_hf_hook", None) is not None and getattr(model._hf_hook, "offload", False):
        raise RuntimeError("STOP: an offload hook was detected")
    return effective_map


def synchronize_both_gpus() -> None:
    for device_index in range(EXPECTED_GPU_COUNT):
        torch.cuda.synchronize(device_index)


def reset_peak_memory_stats() -> None:
    for device_index in range(EXPECTED_GPU_COUNT):
        torch.cuda.reset_peak_memory_stats(device_index)


def peak_memory_gib() -> dict[str, float]:
    gib = 1024 ** 3
    return {
        f"peak_allocated_vram_gpu{index}_gib": torch.cuda.max_memory_allocated(index) / gib
        for index in range(EXPECTED_GPU_COUNT)
    } | {
        f"peak_reserved_vram_gpu{index}_gib": torch.cuda.max_memory_reserved(index) / gib
        for index in range(EXPECTED_GPU_COUNT)
    }


def build_conversation(item: BenchmarkItem) -> list[dict[str, Any]]:
    return [{"role": "user", "content": [{"type": "image", "image": item.image}, {"type": "text", "text": ANNOTATION_PROMPT}]}]


def generate_true_batch(items: list[BenchmarkItem]) -> list[str]:
    conversations = [build_conversation(item) for item in items]
    inputs = processor.apply_chat_template(
        conversations, tokenize=True, add_generation_prompt=True,
        padding=True, return_dict=True, return_tensors="pt",
    )
    inputs.pop("token_type_ids", None)
    input_token_width = inputs["input_ids"].shape[1]
    inputs = inputs.to(INPUT_DEVICE)
    with torch.inference_mode():
        generated_ids = model.generate(**inputs, **GENERATION_PARAMETERS)
    generated_only = generated_ids[:, input_token_width:]
    return processor.batch_decode(generated_only, skip_special_tokens=True, clean_up_tokenization_spaces=False)


def run_configuration(config_id: str, batch_size: int) -> tuple[pd.DataFrame, dict[str, Any]]:
    # Configuration-boundary cleanup is allowed; no per-image or per-batch cache clearing occurs.
    for device_index in range(EXPECTED_GPU_COUNT):
        with torch.cuda.device(device_index):
            torch.cuda.empty_cache()
    warmup_items = list(inference_items[:batch_size])
    try:
        _ = generate_true_batch(warmup_items)
    except torch.cuda.OutOfMemoryError as exc:
        for device_index in range(EXPECTED_GPU_COUNT):
            with torch.cuda.device(device_index):
                torch.cuda.empty_cache()
        raise RuntimeError(f"STOP: configuration {config_id} warm-up OOM; no quantization/offload fallback is permitted") from exc
    synchronize_both_gpus()
    reset_peak_memory_stats()
    synchronize_both_gpus()

    raw_attempts: list[dict[str, Any]] = []
    timed_started = time.perf_counter()
    try:
        for batch_index, offset in enumerate(range(0, len(inference_items), batch_size)):
            batch = list(inference_items[offset:offset + batch_size])
            synchronize_both_gpus()
            batch_started = time.perf_counter()
            raw_outputs = generate_true_batch(batch)
            synchronize_both_gpus()
            batch_seconds = time.perf_counter() - batch_started
            if len(raw_outputs) != len(batch):
                raise RuntimeError("generation returned a different number of responses than inputs")
            for position_in_batch, (item, raw_output) in enumerate(zip(batch, raw_outputs, strict=True)):
                raw_attempts.append({
                    "content_hash": item.content_hash,
                    "sample_position": item.sample_position,
                    "config_id": config_id,
                    "batch_size": batch_size,
                    "batch_index": batch_index,
                    "position_in_batch": position_in_batch,
                    "actual_batch_size": len(batch),
                    "raw_model_output": raw_output,
                    "batch_inference_seconds": batch_seconds,
                    "batch_seconds_per_image": batch_seconds / len(batch),
                })
        synchronize_both_gpus()
    except torch.cuda.OutOfMemoryError as exc:
        for device_index in range(EXPECTED_GPU_COUNT):
            with torch.cuda.device(device_index):
                torch.cuda.empty_cache()
        raise RuntimeError(f"STOP: configuration {config_id} OOM; no quantization/offload fallback is permitted") from exc
    wall_seconds = time.perf_counter() - timed_started
    memory = peak_memory_gib()

    rows: list[dict[str, Any]] = []
    for attempt in raw_attempts:
        raw_output = str(attempt["raw_model_output"]).strip()
        try:
            parsed = parse_model_json(raw_output)
            parse_ok = True
            parse_error = ""
        except (json.JSONDecodeError, TypeError, ValueError) as exc:
            parsed = None
            parse_ok = False
            parse_error = f"{type(exc).__name__}: {exc}"
        rows.append(attempt | {
            "pred_public_relevance": parsed["public_relevance"] if parsed else None,
            "pred_harm_urgency": parsed["harm_urgency"] if parsed else None,
            "pred_vulnerability": parsed["vulnerability"] if parsed else None,
            "pred_sensitivity_score": parsed["sensitivity_score"] if parsed else None,
            "pred_sensitivity_level": parsed["sensitivity_level"] if parsed else None,
            "pred_sensitivity_rationale": parsed["sensitivity_rationale"] if parsed else None,
            "pred_annotation_confidence": parsed["annotation_confidence"] if parsed else None,
            "parse_ok": parse_ok,
            "parse_error": parse_error,
            "raw_model_output": raw_output,
        })
    frame = pd.DataFrame(rows)[PREDICTION_COLUMNS].sort_values("sample_position").reset_index(drop=True)
    if len(frame) != EXPECTED_BENCHMARK_IMAGES or not frame["content_hash"].is_unique:
        raise RuntimeError(f"configuration {config_id} did not produce 27 unique attempts")
    parse_successes = int(frame["parse_ok"].sum())
    summary = {
        "config_id": config_id,
        "model_layout": "one_model_sharded_across_two_gpus",
        "dtype": "float16",
        "max_image_side": MAX_IMAGE_SIDE,
        "batch_size": batch_size,
        "gpu_count": EXPECTED_GPU_COUNT,
        "images_processed": len(frame),
        "wall_clock_seconds": wall_seconds,
        "images_per_second": len(frame) / wall_seconds,
        "seconds_per_image": wall_seconds / len(frame),
        "baseline_seconds_per_image": BASELINE_SECONDS_PER_IMAGE,
        "speedup_vs_baseline": BASELINE_SECONDS_PER_IMAGE / (wall_seconds / len(frame)),
        "parse_successes": parse_successes,
        "parse_failures": len(frame) - parse_successes,
        "parse_success_rate": parse_successes / len(frame),
        "warmup_images": len(warmup_items),
        "warmup_excluded_from_timing": True,
        "timing_scope": "processor/chat-template preparation, CUDA transfer, generation, and batch decoding; in-memory image preload and strict JSON parsing excluded",
    } | memory
    return frame, summary

## 4. Data

Select 9 Gold LOW, 9 MEDIUM, and all 9 HIGH cases with the previous benchmark's deterministic hash ordering. Human labels are discarded before inference; model-facing objects contain only pixels plus an opaque resume key.

In [ ]:
if not GOLD_CSV_PATH.is_file():
    raise FileNotFoundError(f"Gold annotations not found: {GOLD_CSV_PATH}")
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"canonical pilot manifest not found: {MANIFEST_PATH}")
if sha256_file(GOLD_CSV_PATH) != EXPECTED_GOLD_SHA256:
    raise RuntimeError("Gold CSV SHA-256 mismatch")

gold = pd.read_csv(GOLD_CSV_PATH, keep_default_na=False)
if tuple(gold.columns) != GOLD_REQUIRED_COLUMNS:
    raise RuntimeError("Gold CSV columns do not exactly match the frozen schema")
if len(gold) != EXPECTED_GOLD_ROWS:
    raise RuntimeError(f"expected {EXPECTED_GOLD_ROWS} Gold rows, found {len(gold)}")
if gold["content_hash"].isna().any() or not gold["content_hash"].is_unique:
    raise RuntimeError("Gold content_hash values must be non-null and unique")
if set(gold["sensitivity_level"]) != {"low", "medium", "high"}:
    raise RuntimeError("Gold sensitivity levels are not exactly low/medium/high")

selected_parts: list[pd.DataFrame] = []
for level in ("low", "medium", "high"):
    candidates = gold.loc[gold["sensitivity_level"] == level, ["content_hash", "sensitivity_level"]].copy()
    if level == "high" and len(candidates) != BENCHMARK_IMAGES_PER_LEVEL:
        raise RuntimeError(f"expected exactly 9 Gold HIGH cases, found {len(candidates)}")
    if len(candidates) < BENCHMARK_IMAGES_PER_LEVEL:
        raise RuntimeError(f"not enough {level} cases for deterministic sampling")
    purpose = f"select-{level}"
    candidates["selection_key"] = candidates["content_hash"].map(lambda value, purpose=purpose: seeded_order_key(str(value), purpose))
    selected_parts.append(candidates.sort_values(["selection_key", "content_hash"]).head(BENCHMARK_IMAGES_PER_LEVEL))
selected = pd.concat(selected_parts, ignore_index=True)
selected["benchmark_order_key"] = selected["content_hash"].map(lambda value: seeded_order_key(str(value), "benchmark-order"))
selected = selected.sort_values(["benchmark_order_key", "content_hash"]).reset_index(drop=True)
selected["sample_position"] = range(len(selected))
if len(selected) != EXPECTED_BENCHMARK_IMAGES or not selected["content_hash"].is_unique:
    raise RuntimeError("benchmark sample must contain 27 unique hashes")
if selected["sensitivity_level"].value_counts().to_dict() != {"low": 9, "medium": 9, "high": 9}:
    raise RuntimeError("benchmark sample is not balanced 9/9/9")

manifest_records: list[dict[str, Any]] = []
with MANIFEST_PATH.open(encoding="utf-8") as handle:
    for line_number, line in enumerate(handle, start=1):
        if not line.strip():
            raise RuntimeError(f"blank manifest row at line {line_number}")
        record = json.loads(line)
        if not isinstance(record, dict):
            raise TypeError(f"manifest line {line_number} is not an object")
        manifest_records.append(record)
if len(manifest_records) != EXPECTED_MANIFEST_ROWS:
    raise RuntimeError(f"expected 600 manifest rows, found {len(manifest_records)}")

records_by_hash: dict[str, list[dict[str, Any]]] = {}
for record in manifest_records:
    content_hash = record.get("content_hash")
    if not isinstance(content_hash, str) or not content_hash:
        raise RuntimeError("manifest record has an invalid content_hash")
    records_by_hash.setdefault(content_hash, []).append(record)

resolved_rows: list[dict[str, Any]] = []
for row in selected.itertuples(index=False):
    matches = records_by_hash.get(str(row.content_hash), [])
    if len(matches) != 1:
        raise RuntimeError(f"selected hash must resolve exactly once: {row.content_hash} -> {len(matches)}")
    image_path = resolve_manifest_path(matches[0])
    if not image_path.is_file():
        raise FileNotFoundError(f"resolved image is missing: {image_path}")
    resolved_rows.append({"sample_position": int(row.sample_position), "content_hash": str(row.content_hash), "image_path": image_path})
resolved = pd.DataFrame(resolved_rows)
if len(resolved) != EXPECTED_BENCHMARK_IMAGES or not resolved["content_hash"].is_unique or not resolved["image_path"].is_unique:
    raise RuntimeError("resolution must produce 27 unique hashes and canonical image paths")

# Pixels are loaded and resized in memory. Labels, paths, and other metadata do not enter inference objects.
inference_items = tuple(
    BenchmarkItem(int(row.sample_position), str(row.content_hash), load_image_in_memory(Path(row.image_path)))
    for row in resolved.itertuples(index=False)
)
del gold, selected, selected_parts, manifest_records, records_by_hash, resolved, resolved_rows
print(f"Validated and preloaded {len(inference_items)} canonical benchmark images at max side {MAX_IMAGE_SIDE}.")

## 5. Network

Load one unquantized FP16 model with Accelerate's balanced mapping across both T4 GPUs. Loading stops if both GPUs are not used or if any CPU, disk, or meta offload is detected.

In [ ]:
if transformers.__version__ != "5.16.1":
    raise RuntimeError(f"unexpected Transformers version: {transformers.__version__}")
if huggingface_hub.__version__ != "1.29.0":
    raise RuntimeError(f"unexpected huggingface_hub version: {huggingface_hub.__version__}")
if not torch.cuda.is_available() or torch.cuda.device_count() != EXPECTED_GPU_COUNT:
    raise RuntimeError(f"this benchmark requires exactly two CUDA GPUs, found {torch.cuda.device_count()}")
GPU_NAMES = [torch.cuda.get_device_name(index) for index in range(EXPECTED_GPU_COUNT)]
if not all("T4" in name.upper() for name in GPU_NAMES):
    raise RuntimeError(f"this benchmark is specified for Tesla T4 x2, found {GPU_NAMES}")

processor = AutoProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION, trust_remote_code=False)
processor.tokenizer.padding_side = "left"
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

try:
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        torch_dtype=MODEL_DTYPE,
        device_map="balanced",
        max_memory={0: MAX_MEMORY_PER_GPU, 1: MAX_MEMORY_PER_GPU},
        low_cpu_mem_usage=True,
        attn_implementation="sdpa",
        trust_remote_code=False,
    )
except (torch.cuda.OutOfMemoryError, RuntimeError) as exc:
    if isinstance(exc, torch.cuda.OutOfMemoryError) or "out of memory" in str(exc).lower():
        for device_index in range(EXPECTED_GPU_COUNT):
            with torch.cuda.device(device_index):
                torch.cuda.empty_cache()
        raise RuntimeError("STOP: the unquantized FP16 model did not fit across T4 x2 without offload") from exc
    raise
model.eval()
EFFECTIVE_DEVICE_MAP = verify_sharded_model(model)
INPUT_DEVICE = next(model.parameters()).device
if str(INPUT_DEVICE) not in {"cuda:0", "cuda:1"}:
    raise RuntimeError(f"invalid model input device: {INPUT_DEVICE}")
print(json.dumps({"gpu_names": GPU_NAMES, "input_device": str(INPUT_DEVICE), "hf_device_map": EFFECTIVE_DEVICE_MAP}, indent=2))

## 6. Train / Inference Benchmark

There is no training: Qwen weights are never updated. Each configuration receives one excluded warm-up, then processes the same 27 preloaded images. Batch size 2 uses one processor call and one `generate()` call for two independent image/text conversations.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
configuration_frames: dict[str, pd.DataFrame] = {}
configuration_summaries: list[dict[str, Any]] = []
configuration_failures: list[dict[str, Any]] = []

for configuration in BENCHMARK_CONFIGS:
    config_id = str(configuration["config_id"])
    batch_size = int(configuration["batch_size"])
    print(f"Running CONFIG {config_id}: sharded FP16, max side {MAX_IMAGE_SIDE}, batch size {batch_size}")
    try:
        frame, summary = run_configuration(config_id, batch_size)
    except RuntimeError as exc:
        config_b_oom = (
            config_id == "B"
            and "A" in configuration_frames
            and isinstance(exc.__cause__, torch.cuda.OutOfMemoryError)
        )
        if not config_b_oom:
            raise
        configuration_failures.append({
            "config_id": config_id,
            "batch_size": batch_size,
            "status": "oom",
            "error": str(exc),
            "completed_configurations_preserved": sorted(configuration_frames),
        })
        print("CONFIG B ran out of memory; preserving CONFIG A and continuing to summary generation.")
        break
    frame.to_csv(Path(configuration["predictions_path"]), index=False)
    configuration_frames[config_id] = frame
    configuration_summaries.append(summary)
    pd.DataFrame(configuration_summaries).to_csv(BENCHMARK_RESULTS_PATH, index=False)
    display(pd.DataFrame([summary]))

## 7. Evaluation / Benchmark Summary

Summarize throughput, memory, strict parse validity, and cross-batch determinism only. This notebook does not evaluate annotation quality or introduce a pass/fail threshold.

In [ ]:
config_a = configuration_frames["A"].copy()
if "B" in configuration_frames:
    config_b = configuration_frames["B"].copy()
    comparison_columns = [
        "content_hash", "parse_ok", "pred_public_relevance", "pred_harm_urgency",
        "pred_vulnerability", "pred_sensitivity_level",
    ]
    comparison = config_a[comparison_columns].merge(
        config_b[comparison_columns], on="content_hash", how="inner", validate="one_to_one", suffixes=("_a", "_b"),
    )
    if len(comparison) != EXPECTED_BENCHMARK_IMAGES:
        raise RuntimeError("cross-configuration comparison did not match all 27 hashes")
    both_valid = comparison["parse_ok_a"] & comparison["parse_ok_b"]
    triplet_equal = (
        (comparison["pred_public_relevance_a"] == comparison["pred_public_relevance_b"])
        & (comparison["pred_harm_urgency_a"] == comparison["pred_harm_urgency_b"])
        & (comparison["pred_vulnerability_a"] == comparison["pred_vulnerability_b"])
    )
    level_equal = comparison["pred_sensitivity_level_a"] == comparison["pred_sensitivity_level_b"]
    sanity_check = {
        "status": "completed",
        "images_compared": len(comparison),
        "valid_json_in_both": int(both_valid.sum()),
        "identical_component_triplets": int((both_valid & triplet_equal).sum()),
        "different_component_triplets": int((both_valid & ~triplet_equal).sum()),
        "identical_sensitivity_levels": int((both_valid & level_equal).sum()),
        "scope_note": "determinism diagnostic only; not an annotation-quality evaluation",
    }
else:
    if not configuration_failures or configuration_failures[-1]["config_id"] != "B":
        raise RuntimeError("CONFIG B is missing without a recorded OOM")
    sanity_check = {
        "status": "not_run_config_b_oom",
        "images_compared": 0,
        "valid_json_in_both": None,
        "identical_component_triplets": None,
        "different_component_triplets": None,
        "identical_sensitivity_levels": None,
        "scope_note": "CONFIG A preserved; cross-configuration diagnostic unavailable because CONFIG B ran out of memory",
    }

benchmark_results = pd.DataFrame(configuration_summaries)
benchmark_results.to_csv(BENCHMARK_RESULTS_PATH, index=False)
summary_payload = {
    "benchmark_name": "SensiFake Qwen3-VL sharded throughput benchmark",
    "benchmark_status": "completed" if "B" in configuration_frames else "config_b_oom_config_a_preserved",
    "created_at_utc": utc_now(),
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "model_layout": "one complete logical model sharded across cuda:0 and cuda:1",
    "effective_hf_device_map": EFFECTIVE_DEVICE_MAP,
    "input_device": str(INPUT_DEVICE),
    "gpu_names": GPU_NAMES,
    "gpu_count": EXPECTED_GPU_COUNT,
    "dtype": "torch.float16",
    "quantized": False,
    "cpu_offload": False,
    "disk_offload": False,
    "seed": SEED,
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "huggingface_hub_version": huggingface_hub.__version__,
    "cuda_version": torch.version.cuda,
    "generation_parameters": GENERATION_PARAMETERS,
    "image_preprocessing": {"in_memory_only": True, "preserve_aspect_ratio": True, "max_image_side": MAX_IMAGE_SIDE, "resampling": "PIL.Image.Resampling.LANCZOS"},
    "sample_selection": {"method": "seeded SHA-256 ordering reused from previous throughput benchmark", "seed": SEED, "low": 9, "medium": 9, "high": 9, "all_high_used": True},
    "prompt_text": ANNOTATION_PROMPT,
    "baseline": {"total_seconds": BASELINE_TOTAL_SECONDS, "images": BASELINE_IMAGES, "seconds_per_image": BASELINE_SECONDS_PER_IMAGE},
    "configurations": configuration_summaries,
    "configuration_failures": configuration_failures,
    "cross_configuration_sanity_check": sanity_check,
    "total_notebook_runtime_seconds": time.perf_counter() - NOTEBOOK_STARTED_AT,
}
temporary_summary_path = BENCHMARK_SUMMARY_PATH.with_suffix(".json.tmp")
temporary_summary_path.write_text(json.dumps(summary_payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
os.replace(temporary_summary_path, BENCHMARK_SUMMARY_PATH)

display(benchmark_results)
display(pd.DataFrame([sanity_check]))
if configuration_failures:
    print("CONFIG A artifacts and summary were preserved after CONFIG B OOM.")
print(f"Artifacts written to {OUTPUT_DIR}")